# 00 — Tiền xử lý dữ liệu
Notebook này thực hiện toàn bộ quá trình làm sạch và chuẩn bị dữ liệu từ 3 trạm đo mặn.  
Kết quả sẽ được lưu thành file `data/processed/` dùng chung cho các kịch bản.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

DATA_DIR  = Path("data")
SAVE_DIR  = Path("data/processed")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

STATIONS     = ["BenLuc", "CauNoi", "TanAn"]
FEATURE_COLS = ["wind_speed", "temp", "total_precipitation"]


## 1. Đọc dữ liệu thô

In [ ]:
def fix_salinity(series: pd.Series) -> pd.Series:
    """Sửa các lỗi định dạng phổ biến trong cột độ mặn."""
    s = series.astype(str).str.strip()
    s = s.str.replace(",", ".", regex=False)   # 13,8  → 13.8
    s = s.str.replace(r"^\.+", "", regex=True) # .9.5  → 9.5
    s = s.str.replace(r"\.{2,}", ".", regex=True) # 4..2 → 4.2
    return pd.to_numeric(s, errors="coerce")

def load_raw(name: str) -> pd.DataFrame:
    path = DATA_DIR / f"{name}_Weather_2020_2025.csv"
    df = pd.read_csv(path)
    df["Time"] = pd.to_datetime(df["Time"])
    # Loại bỏ duplicate timestamps
    n_dup = df.duplicated(subset="Time").sum()
    df = df.drop_duplicates(subset="Time").sort_values("Time").reset_index(drop=True)
    print(f"[{name}] {len(df)} bản ghi sau khi xoá {n_dup} duplicate")
    return df

raw = {s: load_raw(s) for s in STATIONS}


## 2. Kiểm tra & sửa lỗi cột độ mặn

In [ ]:
for name, df in raw.items():
    sal_col = f"Salinity_{name}"
    bad_mask = pd.to_numeric(df[sal_col].astype(str).str.strip()
                               .str.replace(",", ".", regex=False), errors="coerce").isna()
    bad_vals = df.loc[bad_mask, sal_col].value_counts()
    print(f"[{name}] Giá trị lỗi ({bad_mask.sum()} bản ghi):")
    print(bad_vals.to_string() if len(bad_vals) else "  Không có lỗi")
    print()


In [ ]:
for name in STATIONS:
    sal_col = f"Salinity_{name}"
    raw[name][sal_col] = fix_salinity(raw[name][sal_col])
    print(f"[{name}] Sau sửa — NaN còn lại: {raw[name][sal_col].isna().sum()}")


## 3. Xử lý khoảng trống thời gian (gaps)

In [ ]:
for name, df in raw.items():
    gaps = df["Time"].diff()
    big_gaps = gaps[gaps > pd.Timedelta("2h")]
    print(f"[{name}] Khoảng trống > 2h: {len(big_gaps)} lần "
          f"| Max gap: {gaps.max()}")


In [ ]:
def fill_gaps(df: pd.DataFrame, sal_col: str) -> pd.DataFrame:
    """
    Reindex về lưới 2h đều đặn trong từng năm, sau đó interpolate.
    Không nối xuyên năm để tránh tạo dữ liệu giả giữa các mùa.
    """
    frames = []
    for yr in sorted(df["Time"].dt.year.unique()):
        sub = df[df["Time"].dt.year == yr].copy()
        # tạo lưới 2h từ điểm đầu đến điểm cuối trong năm
        grid = pd.date_range(sub["Time"].min(), sub["Time"].max(), freq="2h")
        sub = sub.set_index("Time").reindex(grid)
        sub.index.name = "Time"
        # interpolate các cột số (giới hạn 5 bước = 10h)
        num_cols = [sal_col] + FEATURE_COLS
        sub[num_cols] = sub[num_cols].interpolate(method="linear", limit=5)
        sub = sub.dropna(subset=num_cols)
        sub = sub.reset_index()
        frames.append(sub)
    return pd.concat(frames, ignore_index=True)

cleaned = {}
for name in STATIONS:
    sal_col = f"Salinity_{name}"
    cleaned[name] = fill_gaps(raw[name], sal_col)
    print(f"[{name}] {len(cleaned[name])} bản ghi sau fill_gaps")


## 4. Thống kê mô tả & visualisation

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=False)
for ax, name in zip(axes, STATIONS):
    sal_col = f"Salinity_{name}"
    df = cleaned[name]
    for yr in sorted(df["Time"].dt.year.unique()):
        sub = df[df["Time"].dt.year == yr]
        ax.plot(sub["Time"], sub[sal_col], linewidth=0.6, label=str(yr))
    ax.set_title(f"Trạm {name} — Độ mặn (‰)")
    ax.legend(fontsize=8, ncol=5)
    ax.set_ylabel("Salinity (‰)")

plt.tight_layout()
plt.savefig("outputs/plots/00_salinity_overview.png", dpi=120)
plt.show()
print("Đã lưu: outputs/plots/00_salinity_overview.png")


In [ ]:
print("\n=== Thống kê mô tả ===")
for name in STATIONS:
    sal_col = f"Salinity_{name}"
    print(f"\n[{name}]")
    print(cleaned[name][[sal_col] + FEATURE_COLS].describe().round(3).to_string())


## 5. Kiểm tra tương quan giữa các trạm

In [ ]:
merged_sal = cleaned["BenLuc"][["Time", "Salinity_BenLuc"]].copy()
for name in ["CauNoi", "TanAn"]:
    merged_sal = pd.merge(merged_sal,
                          cleaned[name][["Time", f"Salinity_{name}"]],
                          on="Time", how="inner")

corr = merged_sal[["Salinity_BenLuc","Salinity_CauNoi","Salinity_TanAn"]].corr()
print("Ma trận tương quan độ mặn giữa các trạm:")
print(corr.round(3))


## 6. Lưu dữ liệu đã xử lý

In [ ]:
for name in STATIONS:
    out_path = SAVE_DIR / f"{name}_clean.csv"
    cleaned[name].to_csv(out_path, index=False)
    print(f"Đã lưu: {out_path}  ({len(cleaned[name])} rows)")

print("\n✅ Tiền xử lý hoàn tất! Dữ liệu sẵn sàng cho các kịch bản.")
